# SSD Dataset Preparation

This notebook prepares the TACO dataset for SSD training by:
1. Consolidamos las 60 clases.
2. Creamos los sets `train`, `val`, `test` igual que se hizo para el modelo YOLO.
3. Re-escalamos las imagenes a 300x300 y generamos las etiquetas en el formato correspondiente.

In [ ]:
!pip install scikit-learn pillow tqdm

In [ ]:
import json
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

# --- CONFIGURACION ---
PROJECT_DIR = Path("/content/Recicling_Project")
TACO_DATA_DIR = PROJECT_DIR / "TACO" / "data"
ANNOTATIONS_JSON = TACO_DATA_DIR / "annotations.json"
SSD_DIR = PROJECT_DIR / "ssd_implementation" / "dataset_ssd"

GROUP_MAP = {
    # AMARILLO
    "Aluminium foil": "amarillo", "Aluminium blister pack": "amarillo", "Carded blister pack": "amarillo",
    "Other plastic bottle": "amarillo", "Clear plastic bottle": "amarillo", "Plastic bottle cap": "amarillo",
    "Metal bottle cap": "amarillo", "Food Can": "amarillo", "Aerosol": "amarillo", "Drink can": "amarillo",
    "Plastic lid": "amarillo", "Metal lid": "amarillo", "Other plastic": "amarillo", "Plastic film": "amarillo",
    "Six pack rings": "amarillo", "Garbage bag": "amarillo", "Other plastic wrapper": "amarillo",
    "Single-use carrier bag": "amarillo", "Polypropylene bag": "amarillo", "Spread tub": "amarillo",
    "Tupperware": "amarillo", "Disposable food container": "amarillo", "Other plastic container": "amarillo",
    "Plastic glooves": "amarillo", "Plastic utensils": "amarillo", "Pop tab": "amarillo",
    "Squeezable tube": "amarillo", "Plastic straw": "amarillo",

    # AZUL
    "Toilet tube": "azul", "Other carton": "azul", "Egg carton": "azul", "Drink carton": "azul",
    "Corrugated carton": "azul", "Meal carton": "azul", "Pizza box": "azul", "Paper cup": "azul",
    "Magazine paper": "azul", "Wrapping paper": "azul", "Normal paper": "azul", "Paper bag": "azul",
    "Paper straw": "azul",

    # VERDE
    "Glass bottle": "verde", "Broken glass": "verde", "Glass cup": "verde", "Glass jar": "verde",

    # MARRON
    "Food waste": "marron",

    # GRIS
    "Battery": "gris", "Foam cup": "gris", "Tissues": "gris", "Plastified paper bag": "gris",
    "Crisp packet": "gris", "Foam food container": "gris", "Rope & strings": "gris",
    "Scrap metal": "gris", "Shoe": "gris", "Styrofoam piece": "gris", "Unlabeled litter": "gris",
    "Cigarette": "gris",
}

class_name_to_idx = {
    "amarillo": 0,
    "azul": 1,
    "verde": 2,
    "marron": 3,
    "gris": 4
}

def make_unique_name(file_name):
    return file_name.replace("/", "__")

def create_pascal_voc_xml(filename, width, height, boxes, output_path):
    annotation = ET.Element("annotation")
    
    folder = ET.SubElement(annotation, "folder")
    folder.text = "images"
    
    fname = ET.SubElement(annotation, "filename")
    fname.text = filename
    
    path_elem = ET.SubElement(annotation, "path")
    path_elem.text = str(output_path.parent.parent / "images" / filename)
    
    source = ET.SubElement(annotation, "source")
    database = ET.SubElement(source, "database")
    database.text = "Unknown"
    
    size = ET.SubElement(annotation, "size")
    w = ET.SubElement(size, "width")
    w.text = str(width)
    h = ET.SubElement(size, "height")
    h.text = str(height)
    d = ET.SubElement(size, "depth")
    d.text = "3"
    
    segmented = ET.SubElement(annotation, "segmented")
    segmented.text = "0"
    
    for box in boxes:
        class_name, xmin, ymin, xmax, ymax = box
        
        obj = ET.SubElement(annotation, "object")
        name = ET.SubElement(obj, "name")
        name.text = class_name
        
        pose = ET.SubElement(obj, "pose")
        pose.text = "Unspecified"
        
        truncated = ET.SubElement(obj, "truncated")
        truncated.text = "0"
        
        difficult = ET.SubElement(obj, "difficult")
        difficult.text = "0"
        
        bndbox = ET.SubElement(obj, "bndbox")
        xmin_elem = ET.SubElement(bndbox, "xmin")
        xmin_elem.text = str(int(xmin))
        ymin_elem = ET.SubElement(bndbox, "ymin")
        ymin_elem.text = str(int(ymin))
        xmax_elem = ET.SubElement(bndbox, "xmax")
        xmax_elem.text = str(int(xmax))
        ymax_elem = ET.SubElement(bndbox, "ymax")
        ymax_elem.text = str(int(ymax))
        
    tree = ET.ElementTree(annotation)
    ET.indent(tree, space="\t", level=0)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)

def process_split(entries, split_name):
    images_dir = SSD_DIR / split_name / "images"
    labels_dir = SSD_DIR / split_name / "labels"
    
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    img_count = 0
    label_count = 0
    
    TARGET_SIZE = 300
    
    for item in tqdm(entries, desc=f"Processing {split_name}"):
        src_img = item["img_path"]
        unique_name = make_unique_name(item["file_name"])
        unique_name_jpg = Path(unique_name).with_suffix(".jpg").name
        dst_img = images_dir / unique_name_jpg
        
        # Procesado de imagenes
        try:
            with Image.open(src_img) as img:
                orig_w, orig_h = img.size
                
                if img.mode != "RGB":
                    img = img.convert("RGB")
                    
                img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.Resampling.LANCZOS)
                img.save(dst_img, "JPEG")
        except Exception as e:
            print(f"Error processing image {src_img}: {e}")
            continue
            
        img_count += 1
        
        scale_x = TARGET_SIZE / orig_w
        scale_y = TARGET_SIZE / orig_h
        
        boxes = []
        for ann in item["annotations"]:
            x, y, w, h = ann["bbox"]
            class_name = class_name_to_idx[ann["class_name"]]  
            
            xmin = x * scale_x
            ymin = y * scale_y
            xmax = (x + w) * scale_x
            ymax = (y + h) * scale_y
            
            xmin = max(0, min(TARGET_SIZE, xmin))
            ymin = max(0, min(TARGET_SIZE, ymin))
            xmax = max(0, min(TARGET_SIZE, xmax))
            ymax = max(0, min(TARGET_SIZE, ymax))
            
            boxes.append((ann["class_name"], xmin, ymin, xmax, ymax))
            
        label_stem = Path(unique_name_jpg).stem
        label_path = labels_dir / f"{label_stem}.xml"
        
        create_pascal_voc_xml(unique_name_jpg, TARGET_SIZE, TARGET_SIZE, boxes, label_path)
        label_count += 1
        
    print(f"[{split_name}] images saved: {img_count}")
    print(f"[{split_name}] labels created: {label_count}")

def main():
    print("Loading annotations...")
    with open(ANNOTATIONS_JSON, "r", encoding="utf-8") as f:
        coco = json.load(f)
        
    categories = coco["categories"]
    cat_id_to_name = {c["id"]: c["name"] for c in categories}
    
    images = coco["images"]
    annotations = coco["annotations"]
    
    img_id_to_info = {img["id"]: img for img in images}
    img_id_to_anns = defaultdict(list)
    
    for ann in annotations:
        img_id_to_anns[ann["image_id"]].append(ann)
        
    dataset_entries = []
    
    for img_id, img_info in img_id_to_info.items():
        file_name = img_info["file_name"]
        img_w = img_info["width"]
        img_h = img_info["height"]
        
        img_path = TACO_DATA_DIR / file_name
        
        if not img_path.exists():
            continue
            
        anns = img_id_to_anns.get(img_id, [])
        valid_anns = []
        
        for ann in anns:
            cat_name = cat_id_to_name[ann["category_id"]]
            
            if cat_name not in GROUP_MAP:
                continue
                
            final_class = GROUP_MAP[cat_name]
            
            valid_anns.append({
                "bbox": ann["bbox"],
                "class_name": final_class
            })
            
        if len(valid_anns) > 0:
            dataset_entries.append({
                "img_path": img_path,
                "file_name": file_name,
                "annotations": valid_anns
            })
            
    print(f"Valid images: {len(dataset_entries)}")
    
    train_entries, temp_entries = train_test_split(
        dataset_entries, test_size=0.2, random_state=42
    )
    val_entries, test_entries = train_test_split(
        temp_entries, test_size=0.5, random_state=42
    )
    
    print(f"Train: {len(train_entries)}")
    print(f"Val: {len(val_entries)}")
    print(f"Test: {len(test_entries)}")
    
    if SSD_DIR.exists():
        print(f"Cleaning existing {SSD_DIR} directory...")
        shutil.rmtree(SSD_DIR)
        
    process_split(train_entries, "train")
    process_split(val_entries, "val")
    process_split(test_entries, "test")

if __name__ == "__main__":
    main()
